# Extended Kalman Filter Formulation

In [1]:
# First install deps
%conda install -c conda-forge numpy scipy sympy matplotlib -y

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/aliaydin/miniforge3/envs/ekf_python

  added / updated specs:
    - matplotlib
    - numpy
    - scipy
    - sympy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    scipy-1.18.1               |  py312h106c528_1        16.5 MB  conda-forge
    ------------------------------------------------------------
                                           Total:        16.5 MB

The following packages will be UPDATED:

  scipy                              1.18.0-py312h54fa4ab_0 --> 1.18.1-py312h106c528_1 



                                                                               
Preparing transaction: done
Verifying transaction: done
Executing transaction: done
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? Yo

In [2]:
import sympy as sp

In [ ]:
# Symbols
dt = sp.symbols('dt')
g = sp.symbols('g')

# Position
p = sp.Matrix([
    sp.Symbol('x'),
    sp.Symbol('y'),
    sp.Symbol('z')
])

# Velocity
v = sp.Matrix([
    sp.Symbol('vx'),
    sp.Symbol('vy'),
    sp.Symbol('vz')
])

# Acceleration measured by IMU
a = sp.Matrix([
    sp.Symbol('ax'),
    sp.Symbol('ay'),
    sp.Symbol('az')
])

# Gravity
gravity = sp.Matrix([
    0,
    0,
    -g
])

# Rotation matrix
# roll, pitch, yaw
phi, theta, psi = sp.symbols('phi theta psi')

Rx = sp.Matrix([
    [1, 0, 0],
    [0, sp.cos(phi), -sp.sin(phi)],
    [0, sp.sin(phi),  sp.cos(phi)]
])

Ry = sp.Matrix([
    [ sp.cos(theta), 0, sp.sin(theta)],
    [0,              1, 0],
    [-sp.sin(theta), 0, sp.cos(theta)]
])

Rz = sp.Matrix([
    [sp.cos(psi), -sp.sin(psi), 0],
    [sp.sin(psi),  sp.cos(psi), 0],
    [0,            0,           1]
])

R = Rz * Ry * Rx
R

Matrix([
[cos(psi)*cos(theta), sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi),  sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi)],
[sin(psi)*cos(theta), sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi), -sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi)],
[        -sin(theta),                              sin(phi)*cos(theta),                               cos(phi)*cos(theta)]])

In [23]:
# position equation
p_next = p + v * dt + sp.Rational(1, 2) * (R * a + gravity) * dt**2
p_next

Matrix([
[ dt**2*(ax*cos(psi)*cos(theta)/2 + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))/2 + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2) + dt*vx + x],
[dt**2*(ax*sin(psi)*cos(theta)/2 + ay*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi))/2 + az*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))/2) + dt*vy + y],
[                                                                 dt**2*(-ax*sin(theta)/2 + ay*sin(phi)*cos(theta)/2 + az*cos(phi)*cos(theta)/2 - g/2) + dt*vz + z]])

In [24]:
# Gyroscope measurements
wx, wy, wz = sp.symbols('wx wy wz')

omega = sp.Matrix([
    wx,
    wy,
    wz
])

# Euler angle rate transformation matrix
E = sp.Matrix([
    [1, 0, sp.sin(theta)],
    [0, sp.cos(phi), -sp.sin(phi)],
    [0, sp.sin(phi), sp.cos(phi)]
])

orientation = sp.Matrix([
    phi,
    theta,
    psi
])


theta_dot = E * omega

# orientation equation
orientation_next = orientation + theta_dot * dt
orientation_next

Matrix([
[         dt*(wx + wz*sin(theta)) + phi],
[dt*(wy*cos(phi) - wz*sin(phi)) + theta],
[  dt*(wy*sin(phi) + wz*cos(phi)) + psi]])

In [25]:
acceleration_world = R * a + gravity

# velocity equation
velocity_next = v + acceleration_world * dt
velocity_next

Matrix([
[ dt*(ax*cos(psi)*cos(theta) + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi)) + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))) + vx],
[dt*(ax*sin(psi)*cos(theta) + ay*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi)) + az*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))) + vy],
[                                                                   dt*(-ax*sin(theta) + ay*sin(phi)*cos(theta) + az*cos(phi)*cos(theta) - g) + vz]])

In [26]:
position = sp.Matrix([
    sp.Symbol('x'),
    sp.Symbol('y'),
    sp.Symbol('z')
])

# Position prediction
position_next = (
    position
    + v * dt
    + sp.Rational(1, 2) * (R * a + gravity) * dt**2
)

# Orientation prediction
orientation_next = (
    orientation
    + E * omega * dt
)

# Velocity prediction
velocity_next = (
    v
    + (R * a + gravity) * dt
)

# Complete predicted state
x_next = sp.Matrix.vstack(
    position_next,
    orientation_next,
    velocity_next
)

x_next

Matrix([
[ dt**2*(ax*cos(psi)*cos(theta)/2 + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))/2 + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2) + dt*vx + x],
[dt**2*(ax*sin(psi)*cos(theta)/2 + ay*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi))/2 + az*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))/2) + dt*vy + y],
[                                                                 dt**2*(-ax*sin(theta)/2 + ay*sin(phi)*cos(theta)/2 + az*cos(phi)*cos(theta)/2 - g/2) + dt*vz + z],
[                                                                                                                                    dt*(wx + wz*sin(theta)) + phi],
[                                                                                                                           dt*(wy*cos(phi) - wz*sin(phi)) + theta],
[                                                                                                                             dt*(wy*sin(phi) + wz*cos(phi)) + psi],
[

In [21]:
x = sp.Matrix([
    position[0], position[1], position[2],
    phi, theta, psi,
    v[0], v[1], v[2]
])

F = x_next.jacobian(x)

# sp.pretty_print(x_next)
F

Matrix([
[1, 0, 0,  dt**2*(ay*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2 + az*(-sin(phi)*sin(theta)*cos(psi) + sin(psi)*cos(phi))/2), dt**2*(-ax*sin(theta)*cos(psi)/2 + ay*sin(phi)*cos(psi)*cos(theta)/2 + az*cos(phi)*cos(psi)*cos(theta)/2), dt**2*(-ax*sin(psi)*cos(theta)/2 + ay*(-sin(phi)*sin(psi)*sin(theta) - cos(phi)*cos(psi))/2 + az*(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))/2), dt,  0,  0],
[0, 1, 0, dt**2*(ay*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))/2 + az*(-sin(phi)*sin(psi)*sin(theta) - cos(phi)*cos(psi))/2), dt**2*(-ax*sin(psi)*sin(theta)/2 + ay*sin(phi)*sin(psi)*cos(theta)/2 + az*sin(psi)*cos(phi)*cos(theta)/2),   dt**2*(ax*cos(psi)*cos(theta)/2 + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))/2 + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2),  0, dt,  0],
[0, 0, 1,                                                                 dt**2*(ay*cos(phi)*cos(theta)/2 - az*sin(phi)*cos(theta)/2),                            dt**2*(-ax*cos(